# MeSH Co-occurrence Network — Topical Structure of the Corpus

This notebook maps the **topical structure** of the corpus through MeSH descriptor co-occurrence: which medical subjects appear together on the same articles, which sit at the centre of the research landscape, and how that structure differs by era.

It builds on the EDA (`03_eda.ipynb`), which established two facts this notebook depends on:
- **Generic descriptors dominate** ("Humans", "Female", "Male", "Adult", …) — they tag nearly every article and carry no topical signal, so they are stripped before any co-occurrence is computed.
- **MeSH depth shifts after ~2019** (mean descriptors/article falls from ~13 to ~8 due to NLM automated indexing), so co-occurrence density is not comparable across that boundary — eras are analysed separately.

Runs on the published metadata (no abstracts needed). The loader mirrors the EDA's Option A / Option B pattern.

## Setup

In [ ]:
import os, glob, collections, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# =====================================================================
# DATA LOADING — pick ONE option (same pattern as 03_eda)
# =====================================================================

# ---- OPTION A: LOCAL (active) ----
ROOT = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(ROOT, "data", "2_clean")

# ---- OPTION B: KAGGLE (uncomment on Kaggle) ----
# _hits = glob.glob("/kaggle/input/*/**/*.parquet", recursive=True) or glob.glob("/kaggle/input/*/*.parquet")
# assert _hits, "no parquet under /kaggle/input — attach the dataset as input"
# DATA_DIR = os.path.dirname(_hits[0])
# =====================================================================

df = pd.read_parquet(DATA_DIR, columns=["uid", "year", "mesh_descriptors"])
df = df[df["year"] <= 2025].copy()                       # drop live edge, as in 03_eda
print(f"loaded {len(df):,} records")
assert len(df) == df["uid"].nunique(), "duplicate PMIDs — dedup did not run"
df.head(3)

## 1. Strip generic descriptors

The most frequent descriptors are demographic/structural, not topical. They co-occur with everything, so leaving them in would make every article look connected and drown out the real topical signal. The set below is removed before any co-occurrence is computed; it is derived from the most-frequent descriptors in the corpus (see EDA §6a), not hand-guessed.

In [ ]:
def diagnose(column, label, threshold=40, top_n=40):
    """Document-frequency diagnostic for a descriptor-list column. Prints the table,
    the coverage-band counts, and plots the top_n. Returns the frequency Series."""
    N = len(df)
    dfq = collections.Counter()
    for lst in df[column]:
        if isinstance(lst, (list, np.ndarray)):
            dfq.update(set(lst))
    pct = (pd.Series(dfq) / N * 100).sort_values(ascending=False)

    print(f"=== {label} — {len(pct):,} distinct descriptors ===")
    print(pct.head(25).round(1).to_string())
    print("\nCoverage bands:")
    for band in [90, 75, 50, 40, 30, 20, 10, 5]:
        print(f"  >= {band:>2}%: {(pct >= band).sum():>4} descriptors")

    plt.figure(figsize=(10, 9))
    top = pct.head(top_n)[::-1]
    sns.barplot(x=top.values, y=top.index, color="#1d6fb8")
    if threshold:
        plt.axvline(threshold, color="#e07a5f", ls="--", lw=1.5, label=f"{threshold}% mark")
        plt.legend()
    plt.title(f"Descriptor document frequency — {label}")
    plt.xlabel("% of articles carrying the descriptor"); plt.ylabel("")
    plt.tight_layout(); plt.show()
    return pct

In [ ]:
def apply_strip(generic_set):
    """Strip generic_set from mesh_descriptors -> mesh_topical, and report the damage."""
    def topical_only(lst):
        if not isinstance(lst, (list, np.ndarray)):
            return []
        return [d for d in lst if d not in generic_set]

    df["mesh_topical"] = df["mesh_descriptors"].map(topical_only)
    df["n_topical"] = df["mesh_topical"].map(len)

    n0 = (df["n_topical"] == 0).sum()
    n1 = (df["n_topical"] == 1).sum()
    n2 = (df["n_topical"] >= 2).sum()
    print(f"removed {len(generic_set)} generic terms. Resulting topical counts:")
    print(f"  0 topical (emptied by the strip): {n0:>9,}  ({n0/len(df)*100:.1f}%)")
    print(f"  1 topical (cannot co-occur):      {n1:>9,}  ({n1/len(df)*100:.1f}%)")
    print(f"  >=2 topical (usable):             {n2:>9,}  ({n2/len(df)*100:.1f}%)")
    print(f"  mean topical/article: {df['n_topical'].mean():.2f}")

In [ ]:
pct_raw = diagnose("mesh_descriptors", "round 1: raw")

In [ ]:
# Justified by the document-frequency table above AND by MeSH category:
# these are demographic check-tags and study-design qualifiers, not topical subjects.
GENERIC = {
    # Check-tags (MeSH administrative population descriptors) — top of the frequency table:
    "Humans",            # 100% — guaranteed by the human-subject filter, zero information
    "Female", "Male",    # 43.6%, 39.0% — sex check-tags
    "Animals", "Mice",   # 15.9%, 5.7% — organism check-tags
    "Pregnancy",         # 3.3% — check-tag
    # Age-band qualifiers (demographic, not topical):
    "Adult", "Middle Aged", "Aged", "Adolescent", "Child", "Young Adult",
    "Aged, 80 and over", "Child, Preschool", "Infant",
    # Study-design / methodology qualifiers (characterize method, not subject):
    "Retrospective Studies", "Prospective Studies", "Cross-Sectional Studies",
    "Follow-Up Studies", "Time Factors", "Surveys and Questionnaires",
    "Treatment Outcome", "Risk Factors",
}

apply_strip(GENERIC)

In [ ]:
pct_stripped = diagnose("mesh_topical", "round 2: after first strip")

In [ ]:
GENERIC |= {
    # epidemiological / study-design qualifiers (method, not subject):
    "Cohort Studies", "Case-Control Studies", "Cross-Sectional Studies",
    "Reproducibility of Results", "Risk Assessment", "Sensitivity and Specificity",
    "Severity of Illness Index", "Prevalence", "Incidence", "Prognosis",
    "Age Factors",                          # demographic qualifier — sibling of the age bands
    "Infant, Newborn",                      # age check-tag — sibling of "Infant"
    "Molecular Sequence Data",              # administrative data tag
    # optional, lean-remove (methods substrates) — decide per your scope:
    # "Cells, Cultured", "Cell Line", "Biomarkers",
}
apply_strip(GENERIC)                        # re-check empties
pct3 = diagnose("mesh_topical", "round 3: after second strip")

**What this shows:** stripping the generic set removes a large share of descriptor mass — these are the terms that carried no topical meaning. Only articles with **two or more** topical descriptors contribute to co-occurrence (a single descriptor has nothing to pair with), so that share is the effective sample for the rest of the notebook.

In [ ]:
# What does "United States" co-occur with? If it pairs with everything ~evenly,
# it's acting as a hub (remove); if it pairs with specific topics (health policy,
# epidemiology), it's topical (keep).
us_pairs = collections.Counter()
for lst in df["mesh_topical"]:
    s = set(lst)
    if "United States" in s:
        us_pairs.update(s - {"United States"})
print("United States co-occurs most with:")
for term, c in us_pairs.most_common(15):
    print(f"  {term:<35} {c:,}")

## 2. Build the co-occurrence counts

For each article, every unordered pair of its topical descriptors is counted once. `itertools.combinations` over the de-duplicated descriptor set per article is the memory-safe way to do this — it never materialises an exploded cross-join of millions of rows.

In [ ]:
from tqdm.auto import tqdm

def cooccurrence(series, show_progress=True):
    """Return (pair_counts, term_freq) over an iterable of descriptor lists."""
    pair = collections.Counter()
    freq = collections.Counter()
    it = tqdm(series, desc="counting co-occurrences", total=len(series)) if show_progress else series
    for lst in it:
        terms = sorted(set(lst))
        freq.update(terms)
        pair.update(itertools.combinations(terms, 2))
    return pair, freq

pair_counts, term_freq = cooccurrence(df["mesh_topical"])
print(f"distinct topical descriptors: {len(term_freq):,}")
print(f"distinct co-occurring pairs:  {len(pair_counts):,}")

In [ ]:
print("max topical descriptors on one article:", df["n_topical"].max())

In [ ]:
counts = np.array(list(pair_counts.values()))
print(f"total pairs: {len(counts):,}")
for thresh in [1, 2, 5, 10, 25, 50, 100, 500, 1000]:
    n = (counts >= thresh).sum()
    print(f"  pairs with count >= {thresh:>4}: {n:>10,}  ({n/len(counts)*100:.1f}%)")

In [ ]:
singletons = (np.array(list(pair_counts.values())) == 1).sum()
print(f"pairs occurring exactly once: {singletons:,} ({singletons/len(pair_counts)*100:.0f}%)")

**What this shows:** the most frequent topical descriptors are the corpus's dominant subjects, and the strongest raw pairs are the subjects that most often appear together. Raw counts favour high-volume topics (a common term pairs with everything), so §3 normalizes to surface genuinely *specific* associations rather than just frequent ones.

## 3. Co-occurrence matrix and normalized association

Raw co-occurrence is dominated by frequency: a common descriptor co-occurs with everything simply because it is common. To find *meaningful* associations, the counts are normalized with the **Jaccard index** — `co(a,b) / (freq(a) + freq(b) − co(a,b))` — which measures how often two terms appear together *relative to* how often either appears at all. High Jaccard = a genuinely specific pairing.

In [ ]:
TOPN = 20
top_terms = [t for t, _ in term_freq.most_common(TOPN)]
idx = {t: i for i, t in enumerate(top_terms)}

raw = np.zeros((TOPN, TOPN))
jac = np.zeros((TOPN, TOPN))
for (a, b), c in pair_counts.items():
    if a in idx and b in idx:
        i, j = idx[a], idx[b]
        raw[i, j] = raw[j, i] = c
        union = term_freq[a] + term_freq[b] - c
        v = c / union if union else 0.0
        jac[i, j] = jac[j, i] = v

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
sns.heatmap(raw, xticklabels=top_terms, yticklabels=top_terms, cmap="Blues",
            ax=axes[0], cbar_kws={"label": "co-occurrence count"})
axes[0].set_title("Raw co-occurrence (top 20 topical descriptors)")
sns.heatmap(jac, xticklabels=top_terms, yticklabels=top_terms, cmap="Reds",
            ax=axes[1], cbar_kws={"label": "Jaccard association"})
axes[1].set_title("Normalized association (Jaccard)")
for ax in axes:
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=8)
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=8)
plt.tight_layout(); plt.show()

**What this shows:** the raw matrix (left) highlights frequent co-occurrences; the Jaccard matrix (right) highlights *specific* ones — pairs that genuinely belong together rather than pairs that are merely both common. Subject areas that are tightly coupled (e.g. a disease and its primary risk factor or treatment) stand out on the right even when their raw counts are modest. Read the right panel for topical structure, the left for volume.

## 4. The co-occurrence network

Treating descriptors as nodes and co-occurrences as weighted edges turns the matrix into a network, where **centrality** identifies the subjects that bridge the most research areas. Edges below a minimum count are dropped to keep the graph readable and to remove incidental pairings. Requires `networkx`.

In [ ]:
try:
    import networkx as nx
except ImportError:
    print("networkx not installed — run `pip install networkx` to build the graph.")
    nx = None

if nx is not None:
    GRAPH_TOPN = 40
    MIN_EDGE = 500          # justified by the pair-count distribution (50% of pairs are singletons)                                  # drop weak/incidental pairs
    graph_terms = set(t for t, _ in term_freq.most_common(GRAPH_TOPN))

    G = nx.Graph()
    for t in graph_terms:
        G.add_node(t, freq=term_freq[t])
    for (a, b), c in pair_counts.items():
        if a in graph_terms and b in graph_terms and c >= MIN_EDGE:
            G.add_edge(a, b, weight=c)
    # keep only the largest connected component for a clean layout
    if G.number_of_nodes():
        comps = sorted(nx.connected_components(G), key=len, reverse=True)
        G = G.subgraph(comps[0]).copy()

    print(f"network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    cent = nx.degree_centrality(G)
    print("\nmost central descriptors (bridge the most subjects):")
    for t, v in sorted(cent.items(), key=lambda x: -x[1])[:8]:
        print(f"  {t:<35} {v:.3f}")

### 4b. Topical communities (clusters of co-occurring subjects)
Rather than computing three-way or higher combinations explicitly, community detection on the pair network finds *groups* of descriptors that all interconnect — the multi-term themes of the corpus. Each community is a set of subjects studied together (e.g. a disease and its mechanisms, or a methods family). Nodes are coloured by community below.

In [ ]:
if nx is not None and G.number_of_nodes():
    from networkx.algorithms.community import greedy_modularity_communities
    communities = list(greedy_modularity_communities(G, weight="weight"))

    print(f"found {len(communities)} topical communities:\n")
    node_comm = {}
    for i, com in enumerate(communities):
        members = sorted(com, key=lambda t: -term_freq[t])
        node_comm.update({n: i for n in com})
        print(f"community {i+1} ({len(com)} terms): {', '.join(members[:8])}"
              + (" …" if len(com) > 8 else ""))

    palette = plt.colormaps["tab10"]
    colors = [palette(node_comm[n] % 10) for n in G.nodes()]

    pos = nx.spring_layout(G, k=0.6, seed=42, weight="weight")
    sizes = [400 + 6000 * cent[n] for n in G.nodes()]
    weights = [G[u][v]["weight"] for u, v in G.edges()]
    wmax = max(weights) if weights else 1
    widths = [0.3 + 3.5 * (w / wmax) for w in weights]

    plt.figure(figsize=(14, 11))
    nx.draw_networkx_edges(G, pos, width=widths, alpha=0.2, edge_color="#888")
    nx.draw_networkx_nodes(G, pos, node_size=sizes, node_color=colors, alpha=0.9)
    nx.draw_networkx_labels(G, pos, font_size=8)
    plt.title("MeSH co-occurrence network, coloured by topical community")
    plt.axis("off"); plt.tight_layout(); plt.show()

**What this shows:** the network resolves into distinct topical communities — each colour is a cluster of subjects that co-occur strongly with each other. These are the corpus's research themes, recovered from pairwise co-occurrence without needing explicit three-way counts: if a group of terms all pair with each other, they fall into one community. The largest, most central node in each cluster is its anchor subject. This is the depth-robust view of topical structure (it reflects which subjects group together, not how many descriptors each paper carries).

**What this shows:** clusters of descriptors that frequently co-occur form the corpus's topical communities; the largest, most central nodes are the subjects that connect the most areas (often broad disease categories or methods that span fields). Edge thickness shows pairing strength. This is descriptive structure, not causation — proximity means "studied together", not "biologically linked".

## 5. How the structure shifts by era

Because MeSH depth drops after ~2019 (EDA §6c), the *number* of descriptors per article — and therefore raw co-occurrence density — is not comparable across that boundary. To compare structure fairly, each era is examined separately and associations are read from the **normalized** (Jaccard) view, which is far less sensitive to depth than raw counts.

In [ ]:
eras = {"2015–2019": (2015, 2019), "2020–2025": (2020, 2025)}

def top_pairs_for(sub, n=10):
    pc, fr = cooccurrence(sub["mesh_topical"])
    scored = []
    for (a, b), c in pc.items():
        union = fr[a] + fr[b] - c
        if union and c >= 20:
            scored.append(((a, b), c / union, c))
    scored.sort(key=lambda x: -x[1])
    return scored[:n]

for label, (lo, hi) in eras.items():
    sub = df[(df["year"] >= lo) & (df["year"] <= hi)]
    print(f"\n=== {label}  ({len(sub):,} articles, mean topical/article {sub['n_topical'].mean():.1f}) ===")
    for (a, b), j, c in top_pairs_for(sub):
        print(f"  {j:.3f}  {a}  +  {b}   (n={c:,})")

**What this shows:** the strongest normalized pairs in each era reveal which subject couplings define that period. Comparing the two lists shows what entered or strengthened — for example, pandemic-era couplings appearing in 2020–2025 that are absent earlier. Because this uses Jaccard (a ratio) rather than raw counts, the post-2019 depth drop does not distort the comparison: it measures *specificity* of association, which is depth-robust, not *volume*, which is not.

## 6. Summary and next steps

### What this notebook produced
- A topical map of the corpus via MeSH co-occurrence, with generic descriptors stripped so the signal is real.
- Normalized (Jaccard) associations that surface specific subject couplings, not just frequent ones.
- A co-occurrence network whose central nodes are the subjects bridging the most research areas.
- An era-split comparison that is robust to the post-2019 MeSH-depth shift.

### Caveats
- **Stripping is a judgement call** — the GENERIC set is the obvious demographic/structural terms; a borderline term ("United States", "Risk Factors") could be argued either way. The set is explicit at the top of §1 so it can be adjusted.
- **Co-occurrence is association, not causation** — terms appear together because they are studied together.
- **No descriptor disambiguation across MeSH revisions** — descriptor names are used as-is; NLM occasionally renames or restructures terms across years.
- **Depth shift** — raw density is not comparable pre/post-2019; rely on the normalized and era-split views for any cross-time claim.

---

## Notebook complete

This notebook built the MeSH co-occurrence network, identified topical clusters and central subjects, and compared topical structure across eras.

**Next Up:** Proceed to `05_coauthorship_network.ipynb` for the co-authorship graph and collaboration structure. Restrict to post-2014 for reliable affiliation coverage, and note that author names are not disambiguated ("J Smith" is not unified across records).